In [40]:
""" Configuration cell"""

from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
import json

import morethemes as mt
from curved_text import curved_text 
mt.set_theme("minimal")


DATA_DIR_profiles = Path("runs/reference/normal")          # where your simulation dumps CSVs
DATA_DIR_inventory = DATA_DIR_profiles 
DATA_DIR_test = Path("runs/reference")           
FIG_DIR  = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

HOURS_TO_SEC = 3600
SEC_TO_HOURS = 1 / HOURS_TO_SEC
M_TO_CM = 1e2

# ----------------------------------------------------------------------
# 1.  Publication-quality matplotlib defaults
# ----------------------------------------------------------------------
LABELS = {"c_T2": r"c_{T_2}", "y_T2": r"y_{T_2}", "aJ_T2": r"aJ_{T_2}",
          "P_g": r"P_g", "a": r"a", "ndot_T2": r"\dot n_{T_2}",
          "n_T2_salt": r"n_{T_2}", "J_T2": r"J_{T_2}", "P_T2": r"P_{T_2}"}

plt.rcParams.update({
    # --- Fonts: Times-compatible to match elsarticle [times] option ---
    "font.family": "serif",
    "font.serif": ["STIXGeneral", "Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    # --- Sizes tuned for Elsevier single-column (88 mm) ---
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 9,
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    # --- Lines, ticks, frame ---
    "axes.linewidth": 0.7,
    "lines.linewidth": 1.3,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": False,
    "ytick.right": False,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "xtick.major.size": 3.5,
    "ytick.major.size": 3.5,
    "xtick.minor.size": 2,
    "ytick.minor.size": 2,
    "xtick.major.width": 0.7,
    "ytick.major.width": 0.7,
    "xtick.minor.width": 0.5,
    "ytick.minor.width": 0.5,
    "legend.frameon": False,
    # --- Output ---
    "savefig.bbox": "tight",
    "savefig.dpi": 300,
    "pdf.fonttype": 42,    # embed fonts as TrueType (Elsevier-friendly)
    "ps.fonttype": 42,
})

In [41]:
# Helpers ---> could move to sparging.helpers
# ----------------------------------------------------------------------
def select_nearest(times, targets):
    """For each target time, return the index of the closest available time."""
    return [int(np.argmin(np.abs(times - t))) for t in targets]

def plot_profile(x, times, data, indices, ylabel, title, fname, colors):
    """
    - x: spatial grid [m], 1D vector
    - times: time grid [s], 1D vector
    - data: 2D array, shape (len(times), len(x))
    - indices: list of time indices to plot
    - ylabel: string for y-axis label
    - title: string for plot title
    - fname: Path to save the figure
    - colors: list of colors for each time index
    
    example usage:
        plot_profile(x, times, y_T2["data"], idx,
                ylabel=r"$y_{T_2}\ \mathrm{[-]}$",
                title=r"T$_2$ molar fraction profile",
                fname=FIG_DIR / "fig1b_y_T2.pdf",
                colors=colors)
    """
    fig, ax = plt.subplots(figsize=(3.5, 2.8))      # single-column width: figsize=(3.5, 2.8), For double-column figures (spanning both columns), use the figure* environment in LaTeX and a matplotlib figsize=(7.2, 3.0)

    for color, i in zip(colors, indices):
        ax.plot(x * 1e2,                            # m -> cm for readability
                data[i],
                color=color,
                label=f"$t = {times[i]/3600:.1f}\\,\\mathrm{{h}}$")

    ax.set_xlabel(r"$x\ \mathrm{[cm]}$")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xlim(x.min() * 1e2, x.max() * 1e2)

    # Scientific notation on y-axis (nice for very small concentrations)
    ax.ticklabel_format(axis="y", style="sci", scilimits=(-2, 2))

    ax.legend(loc="best")
    fig.tight_layout()
    fig.savefig(fname)
    return fig, ax


def _read_units(path):
    """Read the leading '# units: <unit>' comment line, if present."""
    with open(path) as f:
        first = f.readline().strip()
    m = re.match(r"#\s*units:\s*(.*)", first)
    return m.group(1) if m else ""


def load_export(path: Path): # TODO -> move to helpers.py
    """Load an export, inferring its kind from the CSV column layout.
        profile : time-varying profile -> col 0 'x_metres', then one column per time step 't=<s>s'
        static  : time-invariant profile -> columns 'x_metres', 'value'
        series  : time-varying scalar -> columns 't_seconds', 'value'
    """
    unit = _read_units(path)
    df = pd.read_csv(path, comment="#")
    cols = list(df.columns)

    # --- infer kind from structure ---
    if cols[0] == "x_metres" and all(c.startswith("t=") for c in cols[1:]):
        times = np.array(
            [float(re.search(r"t=([\d.eE+-]+)s", c).group(1)) for c in cols[1:]]
        )
        return dict(kind="profile", unit=unit,
                    x=df["x_metres"].to_numpy(),
                    times=times,
                    data=df[cols[1:]].to_numpy().T)
    elif cols == ["x_metres", "value"]:
        return dict(kind="static", unit=unit,
                    x=df["x_metres"].to_numpy(), data=df["value"].to_numpy())
    elif cols == ["t_seconds", "value"]:
        return dict(kind="series", unit=unit,
                    times=df["t_seconds"].to_numpy(), data=df["value"].to_numpy())
    else:
        raise ValueError(f"Unrecognized export layout in {path}: columns={cols}")

def plot_export(path: Path, fractions=None):
    e = load_export(path)
    sym = LABELS.get(path.stem, path.stem)
    fig, ax = plt.subplots(figsize=(3.5, 2.8))

    if e["kind"] == "profile":
        fractions = np.array([0.2, 1, 3, 10]) / 10 if fractions is None else fractions
        idx = select_nearest(e["times"], fractions * e["times"].max())
        colors = plt.cm.viridis(np.linspace(0.05, 0.85, len(idx)))
        for c, i in zip(colors, idx):
            ax.plot(e["x"] * 1e2, e["data"][i], color=c,
                    label=fr"$t = {e['times'][i]/3600:.1f}\,\mathrm{{h}}$")
        ax.set_xlabel(r"$x\ \mathrm{[cm]}$")
        ax.legend(loc="best")

    elif e["kind"] == "static":
        ax.plot(e["x"] * 1e2, e["data"])
        ax.set_xlabel(r"$x\ \mathrm{[cm]}$")

    elif e["kind"] == "series":
        ax.plot(e["times"] / 3600, e["data"])
        ax.set_xlabel(r"$t\ \mathrm{[h]}$")

    ax.set_ylabel(fr"${sym}\ [\mathrm{{{e['unit']}}}]$")
    ax.ticklabel_format(axis="y", style="sci", scilimits=(-2, 2))
    fig.tight_layout()
    return fig, ax

<>:20: SyntaxWarning: invalid escape sequence '\ '
<>:20: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_391189/2832823537.py:20: SyntaxWarning: invalid escape sequence '\ '
  ylabel=r"$y_{T_2}\ \mathrm{[-]}$",


In [42]:
# =====================================================================
# Cell 1 — Figure 1: c_T2, y_T2, J_T2 profiles (stacked, shared x)
# =====================================================================
c_T2 = load_export(DATA_DIR_profiles / "c_T2.csv")
y_T2 = load_export(DATA_DIR_profiles / "y_T2.csv")
J_T2 = load_export(DATA_DIR_profiles / "J_T2.csv")

times = c_T2["times"]
x     = c_T2["x"]

idx = select_nearest(times, np.array([1, 12, 24, 48, 144]) * HOURS_TO_SEC)

# Dark = early (large amplitude)  ->  light = late (decayed): reads as a decay
colors = plt.cm.Greys(np.linspace(0.55,1, len(idx)))

# (dataset, y-axis label) for each panel — quantity identified by ylabel, not title
panels = [
    (c_T2, r"$c_{T_2}\ \mathrm{[mol_{T_2}\,m^{-3}]}$"),
    (y_T2, r"$y_{T_2}\ \mathrm{[-]}$"),
    (J_T2, r"$J_{T_2}\ \mathrm{[mol_{T_2}\,m^{-2}\,s^{-1}]}$"),
]

fig, axes = plt.subplots(3, 1, figsize=(3.5, 6.2), sharex=True)

for ax, (dataset, ylabel) in zip(axes, panels):
    for color, i in zip(colors, idx):
        l, = ax.plot(x * M_TO_CM, dataset["data"][i],
                color=color,
                label=fr"$t = {times[i]*SEC_TO_HOURS:.0f}\,\mathrm{{h}}$")
        curved_text(ax, x * M_TO_CM, dataset["data"][i], text=l.get_label(), offset=6, fontsize=6, color=color)
        
    ax.set_ylabel(ylabel)
    ax.set_xlim(x.min() * M_TO_CM, x.max() * M_TO_CM)
    ax.ticklabel_format(axis="y", style="sci", scilimits=(-2, 2))
    ax.set_ylim(bottom=0)
    if dataset is c_T2:
        ax.set_ylim(top=1.1 * dataset["data"][idx].max())  # leave a little headroom for the curved text

# x-label only on the bottom panel (shared axis)
axes[-1].set_xlabel(r"$z\ \mathrm{[cm]}$")

# panel labels (a) (b) (c) — described in the LaTeX caption
for ax, lab in zip(axes, ["(a)", "(b)", "(c)"]):
    ax.text(0.02, 0.9, lab, transform=ax.transAxes,
            fontweight="bold", va="top", ha="left")

# ONE shared legend (colors mean the same in all panels).
# Placed in panel (a), which is flat and has empty space.
# axes[1].legend(loc="center left", ncol=1, fontsize=7,
#                handlelength=1.5, columnspacing=1.0, labelspacing=0.3)

fig.tight_layout()
fig.subplots_adjust(hspace=0.12)          # snug vertical spacing on shared x
fig.savefig(FIG_DIR / "fig1_profiles.pdf")

In [43]:
# =====================================================================
# Cell 2 — Figure 2: T2 inventory decay + exponential fit
# =====================================================================
n_T2 = load_export(DATA_DIR_inventory / "n_T2_salt.csv")

with open(DATA_DIR_inventory / "summary.json", "r") as f:
    summary = json.load(f)
tau   = summary["fit_summary"]["tau_fitted"]["value"]     # [s]
n0    = summary["fit_summary"]["n0_fitted"]["value"]
tau_h = tau * SEC_TO_HOURS

t_h  = times * SEC_TO_HOURS
data = n_T2["data"]
fit  = n0 * np.exp(-times / tau)

# --- goodness of fit (for the caption) ---
rmse   = np.sqrt(np.mean((data - fit) ** 2))      
rel_rmse   = rmse / n0        # scale-aware
rel_err    = np.abs(data - fit) / data                       # pointwise relative error
max_relerr = rel_err.max()
print(f"RMSE = {rmse:.2e},  relative RMSE = {rel_rmse:.2e},  max relative error = {max_relerr:.2e}, tau = {tau_h:.4f} h, n0 = {n0:.2e}")

n0_e = n0 / np.e                                             # inventory at t = tau

fig, ax = plt.subplots(figsize=(3.5, 2.8))

# Data: thick, pale grey, underneath
ax.plot(t_h, data,
        color="0.6", linewidth=3.0, solid_capstyle="round",
        zorder=1, label="model")

# Fit: thin, dark, dashed, on top
ax.plot(t_h, fit,
        color="red", linewidth=1.0, linestyle="--",
        zorder=2, label=r"$n_{\mathrm{init}}\,e^{-t/\tau}$")

# --- geometric construction of tau ---------------------------------
# vertical segment: axis -> curve, at t = tau
ax.plot([tau_h, tau_h], [0, n0_e],
        color="0.3", linewidth=0.7, linestyle=":", zorder=0)
# horizontal segment: y-axis -> curve, at n = n0/e
ax.plot([t_h.min(), tau_h], [n0_e, n0_e],
        color="0.3", linewidth=0.7, linestyle=":", zorder=0)

# annotation for tau (just above x-axis, next to the vertical line)
ax.annotate(rf"$\tau = {tau_h:.1f}\,\mathrm{{h}}$",
            xy=(tau_h, 0), xytext=(tau_h + 3, 0.06 * n0),
            fontsize=8, color="0.2")
# annotation for n_init/e (just right of y-axis, above the horizontal line)
ax.annotate(r"$\frac{n_{\mathrm{init}}}{e}$",
            xy=(t_h.min(), n0_e), xytext=(tau_h/2, n0_e + 0.02 * n0), ha="right",
            fontsize=8, color="0.2")

ax.set_xlabel(r"$t\ \mathrm{[h]}$")
ax.set_ylabel(r"$n_{T_2}\ \mathrm{[mol_{T_2}]}$")
ax.set_xlim(t_h.min(), t_h.max())
ax.set_ylim(bottom=0)
ax.ticklabel_format(axis="y", style="sci", scilimits=(-2, 2))

# --- inset: pointwise relative error on a log scale -----------------
# axins = ax.inset_axes([0.58, 0.55, 0.38, 0.38])
# axins.semilogy(t_h, rel_err, color="C3", linewidth=1.0)
# axins.set_xlabel(r"$t\,[\mathrm{h}]$", fontsize=6, labelpad=1)
# axins.set_ylabel(r"$\frac{|n_{T_2}-\mathrm{fit}|}{n_{T_2}}$", fontsize=6, labelpad=1)
# axins.tick_params(labelsize=6, pad=1)
# axins.margins(x=0)

# legend: lifted above the decayed tail so it doesn't touch the curve
ax.legend(loc="best", fontsize=8) #bbox_to_anchor=(0.98, 0.14)

fig.tight_layout()
fig.savefig(FIG_DIR / "fig2_inventory.pdf")

RMSE = 2.04e-19,  relative RMSE = 5.61e-08,  max relative error = 1.29e-06, tau = 27.5453 h, n0 = 3.63e-12


In [44]:
# %% ===================== CELL L : load convergence data =====================
""" Convergence study — load data generated by paper/generate_paper_data.py """
import json
import numpy as np

CS_DIR = Path("runs/convergence_study")
_cs = json.load(open(CS_DIR / "convergence_data.json"))
CS_META = _cs["metadata"]
CS_CASES = _cs["cases"]
Bo = CS_META["Bo"]
R_REF = CS_META["refinement_ratio"]

CASE_ORDER = ["low", "mid", "high"]
CASE_LABEL = {
    "low": r"$\Pi \ll 0.1$ (SPP)",
    "mid": r"$\Pi \simeq 0.2$ (LIBRA-Pi)",
    "high": r"$\Pi \gg 0.1$ (PPL)",
}
CASE_COLOR = {"low": "#4477AA", "mid": "#228833", "high": "#CC3311"}
CASE_MARK = {"low": "o", "mid": "s", "high": "^"}


def cs_sweep(case, axis):
    """Return (dt[s], dx[m], tau_fitted[h]) arrays, ordered coarse -> fine."""
    recs = CS_CASES[case][f"{axis}_sweep"]
    dt = np.array([x["dt_s"] for x in recs])
    dx = np.array([x["dx_m"] for x in recs])
    tau = np.array([x["tau_fitted_s"] for x in recs]) * SEC_TO_HOURS
    return dt, dx, tau


def richardson(f, r):
    """Observed order p, extrapolate and GCI from the finest 3 of a coarse->fine
    series (NaN when the triplet is non-monotone). Roache/Celik GCI convention:
    f1 = finest grid, f2 = next-coarsest, f3 = coarsest of the three -- index 1
    is the finest grid by convention, not list/narrative order."""
    f3, f2, f1 = f[-3:]
    eps21, eps32 = f2 - f1, f3 - f2
    if eps32 == 0 or eps21 == 0 or np.sign(eps21) != np.sign(eps32):
        return np.nan, f1, np.nan
    p = np.log(abs(eps32) / abs(eps21)) / np.log(r)
    f_exact = f1 + (f1 - f2) / (r**p - 1)
    gci = 1.25 * abs((f1 - f2) / f1) / (r**p - 1)
    return p, f_exact, gci


# reference tau per case = temporal Richardson extrapolate (dt -> 0)
TAU_REF = {c: richardson(cs_sweep(c, "dt")[2], R_REF)[1] for c in CASE_ORDER}
# spatial reference (dx -> 0)
TAU_REF_X = {c: richardson(cs_sweep(c, "dx")[2], R_REF)[1] for c in CASE_ORDER}
for c in CASE_ORDER:
    print(f"{c:4s}: Pi={CS_CASES[c]['Pi']:.3f}  tau_ref(dt->0)={TAU_REF[c]:.4f} h")


def _ref_slope(ax, xdata, slope, xref, eref, label, color="0.45"):
    """Dashed reference guide of order `slope` through (xref, eref)."""
    x = np.array([xdata.min(), xdata.max()])
    y = eref * (x / xref) ** slope
    ax.plot(x, y, ls="--", lw=0.9, color=color, zorder=0)
    ax.text(x[0] * 1.15, y[0] * 1.4, label, color=color, fontsize=7,
            rotation=0, va="bottom", ha="left")

low : Pi=0.023  tau_ref(dt->0)=24.9156 h
mid : Pi=0.232  tau_ref(dt->0)=27.4733 h
high: Pi=2.324  tau_ref(dt->0)=60.9052 h


In [45]:
# %% ===================== CELL A : log-log refinement =====================
# Two decoupled 1-D refinement studies (temporal | spatial), 3 Pi regimes.
# y = relative error of the fitted tau w.r.t. the Richardson-extrapolated value.
fig, (axt, axx) = plt.subplots(1, 2, figsize=(7.0, 3.0))

for c in CASE_ORDER:
    dt, _, tau = cs_sweep(c, "dt")
    x = dt * SEC_TO_HOURS / TAU_REF[c]                 # dt / tau_ref  (dimensionless)
    e = np.abs(tau - TAU_REF[c]) / TAU_REF[c]
    m = e > 0
    axt.loglog(x[m], e[m], marker=CASE_MARK[c], color=CASE_COLOR[c],
               ms=4, lw=1.1, label=CASE_LABEL[c])
    if c == "mid":
        _ref_slope(axt, x[m], 1, x[m][-1], e[m][-1], r"order 1")

for c in CASE_ORDER:
    _, dx, tau = cs_sweep(c, "dx")
    x = dx * Bo                                         # dx * Bo  [m]
    e = np.abs(tau - TAU_REF_X[c]) / TAU_REF_X[c]
    m = e > 0
    axx.loglog(x[m], e[m], marker=CASE_MARK[c], color=CASE_COLOR[c],
               ms=4, lw=1.1, label=CASE_LABEL[c])
    if c == "mid":
        _ref_slope(axx, x[m], 2, x[m][-1], e[m][-1], r"order 2")

axt.set_xlabel(r"$\Delta t\,/\,\tau_{\mathrm{ref}}$")
axt.set_ylabel(r"$|\tau_{\mathrm{fit}}-\tau_{\mathrm{ref}}|\,/\,\tau_{\mathrm{ref}}$")
axt.set_title("temporal refinement", fontsize=9)
axt.legend(loc="lower right")
axt.text(0.03, 0.94, "(a)", transform=axt.transAxes, fontweight="bold", va="top")

axx.set_xlabel(r"$\Delta x \cdot \mathrm{Bo}\ \ [\mathrm{m}]$")
axx.set_ylabel(r"$|\tau_{\mathrm{fit}}-\tau_{\mathrm{ref}}|\,/\,\tau_{\mathrm{ref}}$")
axx.set_title("spatial refinement", fontsize=9)
axx.text(0.03, 0.94, "(b)", transform=axx.transAxes, fontweight="bold", va="top")

fig.tight_layout()
fig.savefig(FIG_DIR / "convergence_loglog.pdf")

In [46]:
# %% ===================== CELL B : plateau =====================
# fitted tau as relative deviation from the Richardson plateau; shaded band =
# TOL acceptance window
TOL = 0.01  # 1 % acceptance band on tau
fig, (axt, axx) = plt.subplots(1, 2, figsize=(7.0, 3.0), sharey=True)

axt.axhspan(-TOL * 100, TOL * 100, color="0.6", alpha=0.15, zorder=0)
axx.axhspan(-TOL * 100, TOL * 100, color="0.6", alpha=0.15, zorder=0)

for c in CASE_ORDER:
    dt, _, tau = cs_sweep(c, "dt")
    x = dt * SEC_TO_HOURS / TAU_REF[c]
    dev = (tau / TAU_REF[c] - 1) * 100
    axt.semilogx(x, dev, marker=CASE_MARK[c], color=CASE_COLOR[c], ms=4, lw=1.1,
                 label=CASE_LABEL[c])
    # coarsest step within tolerance = recommended dt*
    ok = np.where(np.abs(dev) <= TOL * 100)[0]
    if len(ok):
        i = ok[np.argmax(x[ok])]
        axt.scatter([x[i]], [dev[i]], s=70, facecolors="none",
                    edgecolors=CASE_COLOR[c], linewidths=1.4, zorder=5)

for c in CASE_ORDER:
    _, dx, tau = cs_sweep(c, "dx")
    x = dx * Bo
    dev = (tau / TAU_REF_X[c] - 1) * 100
    axx.semilogx(x, dev, marker=CASE_MARK[c], color=CASE_COLOR[c], ms=4, lw=1.1,
                 label=CASE_LABEL[c])

axt.axhline(0, color="0.4", lw=0.7, ls=":")
axx.axhline(0, color="0.4", lw=0.7, ls=":")
axt.set_ylim(-3.5, 5)
# invert x so refinement runs left -> right into the plateau
axt.invert_xaxis()
axx.invert_xaxis()
axt.set_xlabel(r"$\Delta t\,/\,\tau_{\mathrm{ref}}$  (finer $\rightarrow$)")
axt.set_ylabel(r"$\tau_{\mathrm{fit}}/\tau_{\mathrm{ref}} - 1\ \ [\%]$")
axt.set_title("temporal refinement", fontsize=9)
axt.legend(loc="lower left")
axt.text(0.03, 0.94, "(a)", transform=axt.transAxes, fontweight="bold", va="top")

axx.set_xlabel(r"$\Delta x \cdot \mathrm{Bo}\ \ [\mathrm{m}]$  (finer $\rightarrow$)")
axx.set_title("spatial refinement", fontsize=9)
axx.text(0.03, 0.94, "(b)", transform=axx.transAxes, fontweight="bold", va="top")
axx.text(0.5, 0.5, r"spatial error $< 10^{-5}\,\%$", transform=axx.transAxes,
         ha="center", va="center", fontsize=8, color="0.4", style="italic")

fig.tight_layout()
fig.savefig(FIG_DIR / "convergence_plateau.pdf")

In [47]:
# %% ===================== CELL C : GCI table =====================
import pandas as pd

rows = []
for c in CASE_ORDER:
    _, _, tau_t = cs_sweep(c, "dt")
    _, _, tau_x = cs_sweep(c, "dx")
    pt, ft, gt = richardson(tau_t, R_REF)
    px, fx, gx = richardson(tau_x, R_REF)
    rows.append({
        "case": CASE_LABEL[c],
        "Pi": CS_CASES[c]["Pi"],
        "p_time": pt,
        "tau_exact_h": ft,
        "GCI_time_pct": gt * 100,
        "p_space": px,
        "GCI_space_pct": gx * 100 if not np.isnan(gx) else np.nan,
    })
gci_df = pd.DataFrame(rows).set_index("case")
pd.set_option("display.float_format", lambda v: f"{v:.3g}")
print(gci_df)

# LaTeX fragment consumed by convergence_study.tex via \input (booktabs).
_hdr = {
    "case": "Regime",
    "Pi": r"$\Pi$",
    "p_time": r"$p_{\Delta t}$",
    "tau_exact_h": r"$\tau_{\infty}$ [h]",
    "GCI_time_pct": r"GCI$_{\Delta t}$ [\%]",
    "p_space": r"$p_{\Delta x}$",
    "GCI_space_pct": r"GCI$_{\Delta x}$ [\%]",
}
_ltx = (
    gci_df.reset_index()
    .rename(columns=_hdr)
    .to_latex(
        index=False, escape=False,
        column_format="lrrrrrr",
        formatters={
            r"$\Pi$": lambda v: f"{v:.3g}",
            r"$p_{\Delta t}$": lambda v: f"{v:.2f}",
            r"$\tau_{\infty}$ [h]": lambda v: f"{v:.2f}",
            r"GCI$_{\Delta t}$ [\%]": lambda v: f"{v:.2f}",
            r"$p_{\Delta x}$": lambda v: f"{v:.2f}",
            r"GCI$_{\Delta x}$ [\%]": lambda v: f"{v:.1e}",
        },
        caption=("Grid-convergence verification of the fitted decay time "
                 "$\\tau_{\\mathrm{fit}}$ for the three $\\Pi$ regimes. "
                 "Observed orders $p$ (refinement ratio $r=%g$), Richardson "
                 "extrapolate $\\tau_{\\infty}$, and Roache GCI on the finest "
                 "grid ($\\mathrm{F_s}=1.25$)." % R_REF),
        label="tab:convergence_gci",
        position="htbp",
    )
)
_ltx = _ltx.replace(r"\begin{table}", r"\begin{table*}").replace(r"\end{table}", r"\end{table*}")
(FIG_DIR / "convergence_gci_table.tex").write_text(_ltx)
print("wrote", FIG_DIR / "convergence_gci_table.tex")

                                Pi  p_time  tau_exact_h  GCI_time_pct  \
case                                                                    
$\Pi \ll 0.1$ (SPP)         0.0232   0.998         24.9         0.162   
$\Pi \simeq 0.2$ (LIBRA-Pi)  0.232   0.998         27.5         0.147   
$\Pi \gg 0.1$ (PPL)           2.32   0.999         60.9        0.0662   

                             p_space  GCI_space_pct  
case                                                 
$\Pi \ll 0.1$ (SPP)                2       1.15e-06  
$\Pi \simeq 0.2$ (LIBRA-Pi)        2       1.25e-06  
$\Pi \gg 0.1$ (PPL)                2        1.9e-06  
wrote figures/convergence_gci_table.tex


In [48]:
# %% ===================== CELL D : 2D heatmap (separable error model) =====================
# OPTIONAL / exploratory — not used in the paper.
# joint error reconstructed from the two 1-D studies assuming additivity
# E(dt,dx) = C_t dt^q + C_x dx^p; acceptance region |E|/tau_ref < TOL.
def _fit_power(x, e):
    """least-squares (C, p) for e ~ C x^p over strictly positive e.
    Falls back to a zero component (C=0, p=1) when e is at the noise floor."""
    m = e > e.max() * 1e-3
    if m.sum() < 2:
        return 0.0, 1.0
    p, logC = np.polyfit(np.log(x[m]), np.log(e[m]), 1)
    return np.exp(logC), p


VMAX = 4.5  # shared colour scale across the three panels [%]
fig, axes = plt.subplots(1, 3, figsize=(7.4, 2.7), sharey=True)
levels = np.linspace(0, VMAX, 13)
for ax, c in zip(axes, CASE_ORDER):
    dt, _, tau_t = cs_sweep(c, "dt")
    _, dx, tau_x = cs_sweep(c, "dx")
    dt_h = dt * SEC_TO_HOURS

    Ct, q = _fit_power(dt_h, np.abs(tau_t - TAU_REF[c]))
    Cx, p = _fit_power(dx, np.abs(tau_x - TAU_REF_X[c]))

    # grid over the swept ranges
    xt = np.linspace(dt_h.min(), dt_h.max(), 80) / TAU_REF[c]      # dt/tau_ref
    xx = np.linspace(dx.min(), dx.max(), 80) * Bo                   # dx*Bo
    DT, DX = np.meshgrid(xt * TAU_REF[c], xx / Bo, indexing="ij")
    E = (Ct * DT**q + Cx * DX**p) / TAU_REF[c] * 100               # relative error [%]

    cf = ax.contourf(xt, xx, E.T, levels=levels, cmap="viridis", extend="max")
    cs = ax.contour(xt, xx, E.T, levels=[TOL * 100], colors="w", linewidths=1.4)
    ax.clabel(cs, fmt=lambda v: f"{v:g}%", fontsize=7)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(r"$\Delta t/\tau_{\mathrm{ref}}$")
    ax.set_title(CASE_LABEL[c], fontsize=8)
axes[0].set_ylabel(r"$\Delta x \cdot \mathrm{Bo}$ [m]")
cbar = fig.colorbar(cf, ax=axes, pad=0.015, fraction=0.045)
cbar.set_label(r"$|E_\tau|$  [\% of $\tau_{\mathrm{ref}}$]", fontsize=8)
fig.savefig(FIG_DIR / "convergence_heatmap.pdf")
print("all convergence figures saved")

all convergence figures saved


In [49]:
# %% ===================== CELL M : load 0D validity study data =====================
""" 0D approximation validity study — load data generated by paper/generate_paper_data.py """
import json
import numpy as np
from matplotlib.lines import Line2D

VS_DIR = Path("runs/0D_validity_study")
_vs_meta = json.load(open(VS_DIR / "metadata.json"))
VS = pd.read_csv(VS_DIR / "validity_data.csv")

RMSE_THRESHOLD = _vs_meta["rmse_flag_threshold"]
OTHER_THRESHOLD = _vs_meta["other_group_threshold"]
VS_FLAGGED = (VS.fit_rmse_norm > RMSE_THRESHOLD).to_numpy()

FACTOR_MARK = {"h_l": "o", "K_s": "s"}
BASE_COLOR = "#4477AA"
FLAG_EDGE_COLOR = "#CC3311"
BASE_EDGE_COLOR = "0.25"

print(f"loaded {len(VS)} samples; {int(VS_FLAGGED.sum())} flagged as non-exponential "
      f"(normalized RMSE > {RMSE_THRESHOLD})")


def scatter_by_factor(ax, x, y, other1, other2, factor, flagged, base_color=BASE_COLOR):
    """Scatter y, encoding: marker = factor (h_l/K_s); alpha high when both other
    groups < OTHER_THRESHOLD; edge colour = flagged (fit RMSE > threshold)."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    other1 = np.asarray(other1, dtype=float)
    other2 = np.asarray(other2, dtype=float)
    factor = np.asarray(factor)
    flagged = np.asarray(flagged)
    highlight = (other1 < OTHER_THRESHOLD) & (other2 < OTHER_THRESHOLD)
    for f, marker in FACTOR_MARK.items():
        for hi in (True, False):
            m = (factor == f) & (highlight == hi)
            if not m.any():
                continue
            edgecolors = list(np.where(flagged[m], FLAG_EDGE_COLOR, BASE_EDGE_COLOR))
            ax.scatter(x[m], y[m], marker=marker, facecolors=base_color,
                       edgecolors=edgecolors, linewidths=0.7, s=22,
                       alpha=(0.85 if hi else 0.15), zorder=3 if hi else 2)


def add_style_legend(ax, loc="lower right", **kwargs):
    handles = [
        Line2D([0], [0], marker=FACTOR_MARK["h_l"], linestyle="none",
               markerfacecolor=BASE_COLOR, markeredgecolor=BASE_EDGE_COLOR, label=r"$h_l$ scaled"),
        Line2D([0], [0], marker=FACTOR_MARK["K_s"], linestyle="none",
               markerfacecolor=BASE_COLOR, markeredgecolor=BASE_EDGE_COLOR, label=r"$K_s$ scaled"),
    ]
    ax.legend(handles=handles, loc=loc, fontsize=6, handlelength=1.2,
              labelspacing=0.3, **kwargs)


DISC_ERROR = 5e-3  # time-discretisation floor: dt/(2*tau) at dt = 0.01*tau_pred_ave


def add_error_guides(ax, signed=True, label_x=0.98):
    """Horizontal guides: y=0, +-5% (dotted) and the labelled +-discretisation floor (dashed)."""
    if signed:
        ax.axhline(0, color="0.6", lw=0.6)
    for s in ((1, -1) if signed else (1,)):
        ax.axhline(s * 0.05, color="0.5", lw=0.7, ls=":")
        ax.axhline(s * DISC_ERROR, color="0.5", lw=0.7, ls="--")
    ax.text(label_x, DISC_ERROR, "discretization error", transform=ax.get_yaxis_transform(),
            fontsize=5.5, color="0.35", ha="right" if label_x > 0.5 else "left", va="center",
            bbox=dict(facecolor=plt.rcParams["axes.facecolor"], edgecolor="none", pad=0.5))


loaded 300 samples; 9 flagged as non-exponential (normalized RMSE > 1e-05)


In [50]:
# %% ===================== CELL E : parity plot =====================
# tau_fitted (exponential fit of the inventory decay) vs tau_pred_ave (height-averaged
# analytical prediction). y = x line with a +-10% band; one dot per sample.
fig, ax = plt.subplots(figsize=(3.5, 2.8))

x = VS.tau_pred_ave_s.to_numpy() * SEC_TO_HOURS
y = VS.tau_fitted_s.to_numpy() * SEC_TO_HOURS
edgecolors = list(np.where(VS_FLAGGED, FLAG_EDGE_COLOR, BASE_EDGE_COLOR))
ax.scatter(x, y, s=16, facecolors=BASE_COLOR, edgecolors=edgecolors,
           linewidths=0.6, alpha=0.75, zorder=3)

lims = [min(x.min(), y.min()) * 0.7, max(x.max(), y.max()) * 1.4]
ref = np.array(lims)
ax.plot(ref, ref, color="0.3", lw=1.0, zorder=1, label=r"$y=x$")
ax.plot(ref, 1.1 * ref, color="0.3", lw=0.7, ls="--", zorder=1, label=r"$\pm 10\%$")
ax.plot(ref, 0.9 * ref, color="0.3", lw=0.7, ls="--", zorder=1)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel(r"$\tau_{\mathrm{pred,ave}}\ \mathrm{[h]}$")
ax.set_ylabel(r"$\tau_{\mathrm{fitted}}\ \mathrm{[h]}$")
ax.legend(loc="upper left", fontsize=7)

frac_within = float((np.abs(y - x) / x <= 0.10).mean())
ax.text(0.97, 0.05, f"{frac_within:.0%} within " + r"$\pm10\%$", transform=ax.transAxes,
        fontsize=7, ha="right", color="0.2")

fig.tight_layout()
fig.savefig(FIG_DIR / "validity_parity.pdf")
print(f"parity: {frac_within:.1%} of samples within +-10% (n={len(VS)})")


parity: 57.7% of samples within +-10% (n=300)


In [51]:
# %% ===================== CELL F : error vs Pi =====================
fig, ax = plt.subplots(figsize=(3.5, 2.8))

scatter_by_factor(ax, VS.Pi, VS.e_ave, VS.G_mix_pred, VS.G_P, VS.factor, VS_FLAGGED)

ax.set_xscale("log")
ax.set_yscale("symlog", linthresh=0.01)
ax.set_xlabel(r"$\Pi$")
ax.set_ylabel(r"$e_{\mathrm{ave}} = \dfrac{\tau_{\mathrm{fitted}}-\tau_{\mathrm{pred,ave}}}{\tau_{\mathrm{pred,ave}}}$")
add_error_guides(ax)
add_style_legend(ax)

fig.tight_layout()
fig.savefig(FIG_DIR / "validity_error_vs_Pi.pdf")


In [61]:
# %% ===================== CELL F2 : error vs Pi, PPL-corrected prediction =====================
# tau_pred_corr = tau_pred_ave * Pi/(1-exp(-Pi)), the exact plug-flow gas balance
# factor (-> 1 for Pi << 1, -> Pi for Pi >> 1). Same tau_pred_ave in both series;
# only the Pi entering the correction differs (height-averaged vs bottom).
PI_CORR_COLOR = {"Pi": BASE_COLOR, "Pi_bot": "#EE7733"}
PI_CORR_LABEL = {"Pi": r"$\Pi_{\mathrm{ave}}$", "Pi_bot": r"$\Pi_{\mathrm{bot}}$"}


def ppl_factor(Pi):
    """Pi/(1-exp(-Pi)), written with expm1 to stay accurate as Pi -> 0."""
    Pi = np.asarray(Pi, dtype=float)
    return -Pi / np.expm1(-Pi)


fig, ax = plt.subplots(figsize=(3.5, 2.8))

for col in ("Pi", "Pi_bot"):
    tau_corr = VS.tau_pred_ave_s.to_numpy() * ppl_factor(VS[col])
    e_corr = np.abs((VS.tau_fitted_s.to_numpy() - tau_corr) / tau_corr)
    scatter_by_factor(ax, VS.Pi, e_corr, VS.G_mix_pred, VS.G_P, VS.factor,
                      VS_FLAGGED, base_color=PI_CORR_COLOR[col])
    hi = (VS.Pi > 1).to_numpy()
    print(f"corrected with {col:6s}: median |e| = {np.median(np.abs(e_corr)):.3f} "
          f"(Pi>1: {np.median(np.abs(e_corr[hi])):.3f}), "
          f"{np.mean(np.abs(e_corr) <= 0.10):.0%} within +-10%")

e_ave = VS.e_ave.to_numpy()
hi = (VS.Pi > 1).to_numpy()
print(f"uncorrected (cell F) : median |e| = {np.median(np.abs(e_ave)):.3f} "
      f"(Pi>1: {np.median(np.abs(e_ave[hi])):.3f}), "
      f"{np.mean(np.abs(e_ave) <= 0.10):.0%} within +-10%")

ax.set_xscale("log")
ax.set_yscale("symlog", linthresh=0.01)
ax.set_xlabel(r"$\Pi_{\mathrm{ave}}$")
ax.set_ylabel(r"$e_{\mathrm{corr}} = |\dfrac{\tau_{\mathrm{fitted}}-\tau_{\mathrm{pred,corr}}}"
              r"{\tau_{\mathrm{pred,corr}}}|$")
add_error_guides(ax, signed=False)

color_legend = ax.legend(
    handles=[Line2D([0], [0], marker="o", linestyle="none", markerfacecolor=c,
                    markeredgecolor=BASE_EDGE_COLOR, label="correction with " + PI_CORR_LABEL[k])
             for k, c in PI_CORR_COLOR.items()],
    loc="upper left", fontsize=6, handlelength=1.2, labelspacing=0.3)
ax.add_artist(color_legend)
add_style_legend(ax)

fig.tight_layout()
fig.savefig(FIG_DIR / "validity_error_vs_Pi_corrected.pdf")

corrected with Pi    : median |e| = 0.005 (Pi>1: 0.195), 83% within +-10%
corrected with Pi_bot: median |e| = 0.005 (Pi>1: 0.308), 79% within +-10%
uncorrected (cell F) : median |e| = 0.053 (Pi>1: 3.934), 58% within +-10%


In [ ]:
# %% ===================== CELL F3 : |error| vs Pi, uncorrected vs corrected =====================
# Same errors as cells F/F2 in absolute value on a log axis: height-averaged prediction
# alone vs the same prediction times the PPL factor evaluated with the averaged Pi.
fig, ax = plt.subplots(figsize=(3.5, 2.8))

tau_corr = VS.tau_pred_ave_s.to_numpy() * ppl_factor(VS.Pi)
e_corr = (VS.tau_fitted_s.to_numpy() - tau_corr) / tau_corr

for e, color, label in [
    (VS.e_ave.to_numpy(), "0.6", r"$\tau_{\mathrm{pred,ave}}$"),
    (e_corr, BASE_COLOR, r"$\tau_{\mathrm{pred,ave}}\,\Pi/(1-e^{-\Pi})$"),
]:
    ax.scatter(VS.Pi, np.abs(e), s=16, facecolors=color, edgecolors=BASE_EDGE_COLOR,
               linewidths=0.5, alpha=0.7, label=label, zorder=3)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(r"$\Pi_{\mathrm{ave}}$")
ax.set_ylabel(r"$|e| = |\tau_{\mathrm{fitted}}-\tau_{\mathrm{pred}}|\,/\,\tau_{\mathrm{pred}}$")
add_error_guides(ax, signed=False)
ax.text(0.02, 0.05, r"$5\%$", transform=ax.get_yaxis_transform(), fontsize=5.5,
        color="0.35", ha="left", va="bottom")
ax.legend(loc="upper left", fontsize=6, handlelength=1.2, labelspacing=0.3)

fig.tight_layout()
fig.savefig(FIG_DIR / "validity_abs_error_vs_Pi.pdf")

In [53]:
# %% ===================== CELL G : error vs G_mix (predicted vs actual) =====================
# error vs G_mix: left = G_mix_pred (from tau_pred_ave), right = G_mix (from tau_fitted)
fig, (axp, axa) = plt.subplots(1, 2, figsize=(7.0, 3.0), sharex=True, sharey=True)

scatter_by_factor(axp, VS.G_mix_pred, VS.e_ave, VS.Pi, VS.G_P, VS.factor, VS_FLAGGED)
scatter_by_factor(axa, VS.G_mix, VS.e_ave, VS.Pi, VS.G_P, VS.factor, VS_FLAGGED)

for ax, xlabel, title, lab in [
    (axp, r"$G_{\mathrm{mix,pred}} = (H^2/E_l)\,/\,\tau_{\mathrm{pred,ave}}$", "predicted", "(a)"),
    (axa, r"$G_{\mathrm{mix}} = (H^2/E_l)\,/\,\tau_{\mathrm{fitted}}$", "actual", "(b)"),
]:
    ax.set_xscale("log")
    ax.set_xlabel(xlabel, fontsize=8)
    ax.set_title(title, fontsize=9)
    ax.text(0.03, 0.94, lab, transform=ax.transAxes, fontweight="bold", va="top")

axp.set_yscale("symlog", linthresh=0.01)
axp.set_ylabel(r"$e_{\mathrm{ave}} = \dfrac{\tau_{\mathrm{fitted}}-\tau_{\mathrm{pred,ave}}}{\tau_{\mathrm{pred,ave}}}$")
for ax in (axp, axa):
    add_error_guides(ax)

add_style_legend(axa, loc="lower right")
fig.tight_layout()
fig.savefig(FIG_DIR / "validity_error_vs_Gmix.pdf")


In [54]:
# %% ===================== CELL H : error vs G_P (bottom vs height-averaged) =====================
# error vs G_P: left = e_bot (bottom prediction), right = e_ave (height-averaged)
fig, (axb, axa) = plt.subplots(1, 2, figsize=(7.0, 3.0), sharex=True, sharey=True)

scatter_by_factor(axb, VS.G_P, VS.e_bot, VS.Pi, VS.G_mix_pred, VS.factor, VS_FLAGGED)
scatter_by_factor(axa, VS.G_P, VS.e_ave, VS.Pi, VS.G_mix_pred, VS.factor, VS_FLAGGED)

axb.set_ylabel(r"$e_{\mathrm{bot}} = \dfrac{\tau_{\mathrm{fitted}}-\tau_{\mathrm{pred,bot}}}{\tau_{\mathrm{pred,bot}}}$")
axa.set_ylabel(r"$e_{\mathrm{ave}} = \dfrac{\tau_{\mathrm{fitted}}-\tau_{\mathrm{pred,ave}}}{\tau_{\mathrm{pred,ave}}}$")
for ax, title, lab in [(axb, "bottom-evaluated", "(a)"), (axa, "height-averaged", "(b)")]:
    ax.set_xscale("log")
    ax.set_yscale("symlog", linthresh=0.01)
    ax.set_xlabel(r"$G_P$")
    ax.set_title(title, fontsize=9)
    ax.text(0.03, 0.94, lab, transform=ax.transAxes, fontweight="bold", va="top")
    add_error_guides(ax)

add_style_legend(axa, loc="lower right")
fig.tight_layout()
fig.savefig(FIG_DIR / "validity_error_vs_GP.pdf")


In [55]:
# %% ===================== CELL S : 0D validity study — text summary =====================
def _approx_threshold(x, y_abs, mask, target=0.05):
    """Largest x (ascending, within `mask`) below which y_abs stays under `target`."""
    xs = np.asarray(x)[mask]
    ys = np.asarray(y_abs)[mask]
    order = np.argsort(xs)
    xs, ys = xs[order], ys[order]
    below = ys < target
    if below.all():
        return xs[-1]
    first_above = np.argmax(~below)
    return xs[first_above - 1] if first_above > 0 else float("nan")


highlight_Pi = (VS.G_mix_pred < OTHER_THRESHOLD) & (VS.G_P < OTHER_THRESHOLD)
highlight_Gmix = (VS.Pi < OTHER_THRESHOLD) & (VS.G_P < OTHER_THRESHOLD)
highlight_GP = (VS.Pi < OTHER_THRESHOLD) & (VS.G_mix_pred < OTHER_THRESHOLD)

Pi_threshold = _approx_threshold(VS.Pi, VS.e_ave.abs(), highlight_Pi.to_numpy())
Gmix_threshold = _approx_threshold(VS.G_mix_pred, VS.e_ave.abs(), highlight_Gmix.to_numpy())

# e_bot changes sign with G_P; report the sign crossover
gp_sorted = VS.G_P[highlight_GP.to_numpy()].sort_values()
ebot_sorted = VS.e_bot[gp_sorted.index]
sign_change = np.where(np.diff(np.sign(ebot_sorted)) != 0)[0]
GP_crossover = float(gp_sorted.iloc[sign_change[0]]) if len(sign_change) else float("nan")
eave_slope_hi = VS.e_ave[highlight_GP.to_numpy() & (VS.G_P >= OTHER_THRESHOLD)].median()
eave_slope_lo = VS.e_ave[highlight_GP.to_numpy() & (VS.G_P < OTHER_THRESHOLD)].median()

hl_mask = (VS.factor == "h_l").to_numpy()
Ks_mask = (VS.factor == "K_s").to_numpy()

# compare h_l vs K_s by regressing out Pi (log-log on |e_ave|) and comparing
# residuals, not raw medians
log_Pi = np.log(VS.Pi.to_numpy())
log_e = np.log(VS.e_ave.abs().to_numpy())
slope, intercept = np.polyfit(log_Pi, log_e, 1)
residual = log_e - (slope * log_Pi + intercept)
med_resid_hl = np.median(residual[hl_mask])
med_resid_Ks = np.median(residual[Ks_mask])

print("=== 0D validity study summary (n=%d) ===" % len(VS))
print(f"- parity: {frac_within:.1%} of samples within +-10% of tau_fitted = tau_pred_ave")
print(f"- approx. threshold Pi        < {Pi_threshold:.3g}   -> |e_ave| < 5% (within the "
      f"highlighted, other-groups-small population)")
print(f"- approx. threshold G_mix_pred< {Gmix_threshold:.3g} -> |e_ave| < 5% (within the "
      f"highlighted, other-groups-small population)")
print(f"- figure 4: e_bot changes SIGN near G_P~{GP_crossover:.3g} (small positive below, "
      f"increasingly negative above -- tau_pred_bot underestimates then overestimates "
      f"tau_fitted); e_ave stays positive throughout and grows much more gently "
      f"(median {eave_slope_lo:.3f} for G_P<{OTHER_THRESHOLD} vs {eave_slope_hi:.3f} for "
      f"G_P>={OTHER_THRESHOLD}) -- height-averaging reduces but does not eliminate the "
      f"G_P-driven error; a plain |e| comparison would have hidden the sign crossover")
print(f"- Pi-controlled test (median log|e_ave| residual after regressing out log(Pi), "
      f"slope={slope:.2f}): {med_resid_hl:+.3f} (h_l, n={int(hl_mask.sum())}) vs "
      f"{med_resid_Ks:+.3f} (K_s, n={int(Ks_mask.sum())}) -- "
      f"{'consistent with' if abs(med_resid_hl - med_resid_Ks) < 0.1 else 'inconsistent with'} "
      f"pure Pi-controlled behaviour (residuals near 0 and equal for both => the two "
      f"populations sit on the same e_ave-vs-Pi curve, see figure 2)")
print(f"- {int(VS_FLAGGED.sum())}/{len(VS)} samples flagged with normalized RMSE > "
      f"{RMSE_THRESHOLD} (non-exponential decay)")


=== 0D validity study summary (n=300) ===
- parity: 57.7% of samples within +-10% of tau_fitted = tau_pred_ave
- approx. threshold Pi        < 0.0877   -> |e_ave| < 5% (within the highlighted, other-groups-small population)
- approx. threshold G_mix_pred< 1.91e-06 -> |e_ave| < 5% (within the highlighted, other-groups-small population)
- figure 4: e_bot changes SIGN near G_P~0.0163 (small positive below, increasingly negative above -- tau_pred_bot underestimates then overestimates tau_fitted); e_ave stays positive throughout and grows much more gently (median 0.007 for G_P<0.1 vs 0.021 for G_P>=0.1) -- height-averaging reduces but does not eliminate the G_P-driven error; a plain |e| comparison would have hidden the sign crossover
- Pi-controlled test (median log|e_ave| residual after regressing out log(Pi), slope=0.72): -0.329 (h_l, n=158) vs -0.388 (K_s, n=142) -- consistent with pure Pi-controlled behaviour (residuals near 0 and equal for both => the two populations sit on the same e_

In [56]:
# %% ===================== CELL N : load Sobol study data =====================
""" Sobol sensitivity study — load data generated by paper/generate_paper_data.py """
import json
import numpy as np
from scipy.stats import sobol_indices

# --- select which Sobol dataset to load (the plots adapt via its metadata) ---
SB_DIR = Path("runs/sobol_input_params")       # study 1: gas flow 50-1000 log-uniform,  N=512
# SB_DIR = Path("runs/sobol_input_params_2")   # study 2: gas flow 300-1000 uniform,     N=256
# SB_DIR = Path("runs/sobol_input_params_3")   # study 3: gas flow 300-1000 log-uniform, N=256

_sb_meta = json.load(open(SB_DIR / "metadata.json"))
SB = pd.read_csv(SB_DIR / "sobol_data.csv")
# per-dataset figure suffix ("", "_2", ...) so studies do not overwrite each other
SB_SUFFIX = SB_DIR.name.replace("sobol_input_params", "")

SB_PARAMS = _sb_meta["param_space"]           # ordered list of {name, unit, min, max, scale, ...}
SB_NAMES = [p["name"] for p in SB_PARAMS]
SB_N = _sb_meta["n_base_samples"]
SB_D = _sb_meta["d"]
SB_QOI = _sb_meta["qoi"]                       # "tau_fitted_s"

SB_LABEL = {                                   # short math labels for the plots
    "temperature": r"$T$",
    "gas_flow": r"$\dot{n}_g$",
    "top_pressure": r"$P_\mathrm{top}$",
    "nozzle_diameter": r"$d_\mathrm{noz}$",
}
SB_COLOR = "#4477AA"


def sobol_matrices(df, n=None, qoi=SB_QOI):
    """Regroup the tagged rows into the {f_A, f_B, f_AB} dict for
    scipy.stats.sobol_indices, using the first `n` base samples (Sobol prefix)."""
    n = n or SB_N
    yA = np.full(n, np.nan)
    yB = np.full(n, np.nan)
    yAB = np.full((SB_D, n), np.nan)
    for role, j, i, val in zip(df.design_role, df.base_index, df.var_index, df[qoi]):
        j = int(j)
        if j >= n:
            continue
        if role == "A":
            yA[j] = val
        elif role == "B":
            yB[j] = val
        else:
            yAB[int(i), j] = val
    assert np.isfinite(yA).all() and np.isfinite(yB).all() and np.isfinite(yAB).all(), (
        f"incomplete Saltelli design for n={n}"
    )
    return {"f_A": yA[None, :], "f_B": yB[None, :], "f_AB": yAB[:, None, :]}


print(f"loaded {len(SB)} runs; N_base={SB_N}, d={SB_D}, QoI={SB_QOI}")
print(f"tau_fitted range: {SB[SB_QOI].min() / 3600:.1f} - {SB[SB_QOI].max() / 3600:.1f} h")

loaded 3072 runs; N_base=512, d=4, QoI=tau_fitted_s
tau_fitted range: 9.8 - 264.3 h


In [57]:
# %% ===================== CELL I : total-order Sobol indices =====================
# Total-order S_Ti (bars, bootstrap CIs) and first-order S_i (markers) per input.
res = sobol_indices(func=sobol_matrices(SB), n=SB_N)
ci = res.bootstrap(n_resamples=999)
S_T = res.total_order.ravel()
S_1 = res.first_order.ravel()
ST_lo = ci.total_order.confidence_interval.low.ravel()
ST_hi = ci.total_order.confidence_interval.high.ravel()

order = np.argsort(S_T)[::-1]                  # most influential first
xpos = np.arange(SB_D)
# CI half-widths (NaN CIs of near-zero indices -> no bar)
yerr = np.vstack([S_T[order] - ST_lo[order], ST_hi[order] - S_T[order]])
yerr = np.nan_to_num(np.clip(yerr, 0, None))

fig, ax = plt.subplots(figsize=(3.5, 2.8))
ax.bar(xpos, S_T[order], width=0.62, color=SB_COLOR, edgecolor="0.25",
       yerr=yerr, capsize=3, error_kw=dict(lw=0.8), zorder=2,
       label=r"total order $S_{T_i}$")
ax.plot(xpos, S_1[order], "D", ms=5, color="#CC3311", zorder=3,
        label=r"first order $S_i$")
ax.set_xticks(xpos)
ax.set_xticklabels([SB_LABEL[SB_NAMES[k]] for k in order])
ax.tick_params(axis="x", which="minor", bottom=False)  # categorical axis: no minor ticks
ax.set_ylabel("Sobol index")
ax.set_ylim(0, max(1.0, float(np.nanmax(ST_hi)) * 1.1))
ax.axhline(0, color="0.6", lw=0.6)
ax.legend(loc="upper right", fontsize=7)
fig.tight_layout()
fig.savefig(FIG_DIR / f"sobol_indices{SB_SUFFIX}.pdf")
for k in order:
    print(f"  {SB_NAMES[k]:16s}: S_T={S_T[k]:.3f} [{ST_lo[k]:.3f}, {ST_hi[k]:.3f}]  S_1={S_1[k]:.3f}")

  gas_flow        : S_T=0.737 [0.648, 0.843]  S_1=0.696
  temperature     : S_T=0.214 [0.179, 0.252]  S_1=0.179
  nozzle_diameter : S_T=0.049 [0.041, 0.062]  S_1=0.037
  top_pressure    : S_T=0.048 [0.040, 0.058]  S_1=0.030


In [58]:
# %% ===================== CELL J : Sobol convergence with sample size =============
# Total-order indices vs N (nested Sobol prefixes from metadata), with bootstrap
# 95% CI bands showing the sampling uncertainty shrink as N grows.
subsets = _sb_meta["convergence_subsets"]
ST_conv = np.zeros((len(subsets), SB_D))
ST_lo_conv = np.zeros((len(subsets), SB_D))
ST_hi_conv = np.zeros((len(subsets), SB_D))
for r_i, n in enumerate(subsets):
    res_n = sobol_indices(func=sobol_matrices(SB, n=n), n=n)
    ci_n = res_n.bootstrap(n_resamples=999)
    ST_conv[r_i] = res_n.total_order.ravel()
    ST_lo_conv[r_i] = ci_n.total_order.confidence_interval.low.ravel()
    ST_hi_conv[r_i] = ci_n.total_order.confidence_interval.high.ravel()
# NaN CIs of near-zero indices -> collapse band to the point
ST_lo_conv = np.where(np.isfinite(ST_lo_conv), ST_lo_conv, ST_conv)
ST_hi_conv = np.where(np.isfinite(ST_hi_conv), ST_hi_conv, ST_conv)

fig, ax = plt.subplots(figsize=(3.5, 2.8))
colors = plt.cm.viridis(np.linspace(0.08, 0.82, SB_D))
for k in range(SB_D):
    ax.fill_between(subsets, ST_lo_conv[:, k], ST_hi_conv[:, k],
                    color=colors[k], alpha=0.18, lw=0)
    ax.plot(subsets, ST_conv[:, k], "-o", ms=4, color=colors[k],
            label=SB_LABEL[SB_NAMES[k]])
ax.axhline(1.0, color="0.6", lw=0.6, ls=":")  # theoretical max
ax.set_xscale("log", base=2)
ax.set_xticks(subsets)
ax.set_xticklabels([str(n) for n in subsets])
ax.set_xlabel(r"base samples $N$")
ax.set_ylabel(r"total-order index $S_{T_i}$")
ax.set_ylim(0, max(1.05, float(np.nanmax(ST_hi_conv)) * 1.08))
ax.legend(loc="upper right", fontsize=7, ncol=2)
fig.tight_layout()
fig.savefig(FIG_DIR / f"sobol_convergence{SB_SUFFIX}.pdf")
print("converged S_T at N=%d:" % subsets[-1],
      {SB_NAMES[k]: round(float(ST_conv[-1, k]), 3) for k in range(SB_D)})
print("95%% CI width  N=%d -> N=%d:" % (subsets[0], subsets[-1]),
      {SB_NAMES[k]: (round(float(ST_hi_conv[0, k] - ST_lo_conv[0, k]), 3),
                     round(float(ST_hi_conv[-1, k] - ST_lo_conv[-1, k]), 3))
       for k in range(SB_D)})

converged S_T at N=512: {'temperature': 0.214, 'gas_flow': 0.737, 'top_pressure': 0.048, 'nozzle_diameter': 0.049}
95% CI width  N=8 -> N=512: {'temperature': (0.969, 0.072), 'gas_flow': (1.353, 0.183), 'top_pressure': (0.03, 0.019), 'nozzle_diameter': (0.122, 0.022)}


In [59]:
# %% ===================== CELL K : operating-point map (tau vs T & gas flow) =====
# Operating-point tool: fitted tau over the (gas flow, temperature) plane from the
# largest Sobol dataset (study 1). Colour = tau (log): pale/fast -> deep red/slow;
# isolines mark equal-tau couples. tau surface = 4-D RBF fit evaluated at nominal
# pressure & nozzle.
from matplotlib.colors import LogNorm
from scipy.interpolate import RBFInterpolator

OP_DIR = Path("runs/sobol_input_params")            # largest dataset (study 1)
OP = pd.read_csv(OP_DIR / "sobol_data.csv")
P_NOM, D_NOM = 1.2, 2.0                             # LIBRA-Pi design pressure & nozzle
NOMINAL = dict(flow=500, T=550)                     # nominal operating point

T = OP.temperature_C.to_numpy()
F = OP.gas_flow_sccm.to_numpy()
tau = OP.tau_fitted_s.to_numpy() / 3600.0

# 4-D RBF -> log10(tau), evaluated on the (flow, T) grid at fixed design P & nozzle
X = np.column_stack([T, np.log10(F), OP.top_pressure_atm, np.log10(OP.nozzle_diameter_mm)])
mu, sd = X.mean(0), X.std(0)
rbf = RBFInterpolator((X - mu) / sd, np.log10(tau),
                      kernel="thin_plate_spline", smoothing=len(X) * 0.01)

ng = 200
Tg = np.linspace(T.min(), T.max(), ng)
Fg = np.logspace(np.log10(F.min()), np.log10(F.max()), ng)
TT, FF = np.meshgrid(Tg, Fg, indexing="ij")
grid = np.column_stack([TT.ravel(), np.log10(FF).ravel(),
                        np.full(TT.size, P_NOM), np.full(TT.size, np.log10(D_NOM))])
tau_grid = 10 ** rbf((grid - mu) / sd).reshape(TT.shape)

fig, ax = plt.subplots(figsize=(4.4, 3.3))
norm = LogNorm(vmin=tau_grid.min(), vmax=tau_grid.max())
pcm = ax.pcolormesh(FF, TT, tau_grid, cmap="YlOrRd", norm=norm, shading="gouraud")
# faint sample coverage
ax.scatter(F, T, s=1.5, c="0.35", alpha=0.15, linewidths=0, zorder=1)
# isolines of equal tau
levels = [h for h in (12, 18, 24, 36, 48, 72, 96, 144, 192) if tau_grid.min() < h < tau_grid.max()]
cs = ax.contour(FF, TT, tau_grid, levels=levels, colors="0.15", linewidths=0.7, zorder=2)
ax.clabel(cs, fmt=lambda v: f"{v:g} h", fontsize=6, inline=True)
# nominal LIBRA-Pi operating point
ax.plot(NOMINAL["flow"], NOMINAL["T"], marker="*", ms=14, mfc="white",
        mec="0.1", mew=0.9, zorder=5)
ax.annotate("nominal", (NOMINAL["flow"], NOMINAL["T"]), textcoords="offset points",
            xytext=(7, 5), fontsize=6.5, color="0.1")

ax.set_xscale("log")
ax.set_xticks([50, 100, 200, 500, 1000])
ax.get_xaxis().set_major_formatter(plt.matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:g}"))
ax.set_xlim(F.min(), F.max())
ax.set_ylim(T.min(), T.max())
ax.set_xlabel(r"gas flow rate  $\dot{n}_g$  [sccm]")
ax.set_ylabel(r"temperature  [$^\circ$C]")
cbar = fig.colorbar(pcm, ax=ax, pad=0.02)
cbar.set_label(r"extraction time  $\tau$  [h]")
cbar.ax.text(0.5, 1.03, "slower", transform=cbar.ax.transAxes, ha="center", va="bottom",
             fontsize=6.5, color="0.3")
cbar.ax.text(0.5, -0.03, "faster", transform=cbar.ax.transAxes, ha="center", va="top",
             fontsize=6.5, color="0.3")
fig.tight_layout()
fig.savefig(FIG_DIR / "operating_point_map.pdf")
print(f"tau at nominal ({NOMINAL['flow']} sccm, {NOMINAL['T']} C): "
      f"{10 ** rbf(((np.array([NOMINAL['T'], np.log10(NOMINAL['flow']), P_NOM, np.log10(D_NOM)]) - mu) / sd)[None])[0]:.1f} h")

tau at nominal (500 sccm, 550 C): 27.7 h
